In [ ]:
# Portable data paths; execute notebooks from the package root.
import os
from pathlib import Path
_review_base = Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()
DATA_ROOT = Path(os.environ.get("REVIEW_DATA_ROOT", str(_review_base / "data"))).resolve()
RESULTS_ROOT = Path(os.environ.get("REVIEW_RESULTS_ROOT", str(_review_base / "results"))).resolve()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


### Supercubic sampling — 12 cases (含 Case 11/12 浸没冷却 DLC)

In [ ]:
import argparse
import os

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from scipy.stats import qmc


In [ ]:
# Table_B1 现在改为读取新版对照表（含 Case 1-12，其中 11/12 是浸没冷却 DLC 版本）
TABLE_PATH = str(DATA_ROOT / "Table_B1_对照表_补充DLC.xlsx")
SHEET_NAME  = "Table_B1_代码可用(转换后)"
N_CASES     = 12
FIRST_DATA_ROW  = 5      # 代码可用sheet 参数数据从第5行开始（第2-4行是表头）
CASE_COL_START  = 5      # E列开始是 Case 1，一路到 Case 10(N列)，Case 1-DLC(O列)=Case11，Case 2-DLC(P列)=Case12
CODE_VAR_COL    = 2      # B列是代码变量名


In [ ]:
# ----------------------------------------------------------------------
# 每个 case 对应的函数、以及 w[] 向量里(除去前3位气候变量 T_oa/RH_oa/P_oa 后)
# 参数的精确顺序。顺序取自 simulation_funs_DC.py 源码里各函数对 w[i] 的定义，
# 一一核对过，不是凭猜测排的。
#
# 注意：Case 5/Case 8 都调用 PUE_WUE_Chiller（中型/小型只是取值区间不同）；
#      Case 7/Case 9 都调用 PUE_WUE_AIRChiller，同理。
#      函数名不是 case 的唯一标识，取值区间才是。
# ----------------------------------------------------------------------

CASE_FUNC_NAME = {
    1:  "PUE_WUE_AE_Chiller",
    2:  "PUE_WUE_Chiller_Watereconomier",
    3:  "PUE_WUE_AE_Chiller_Colo",
    4:  "PUE_WUE_WE_Chiller_Colo",
    5:  "PUE_WUE_Chiller",                           # midsize
    6:  "PUE_WUE_AE_AIRChiller",
    7:  "PUE_WUE_AIRChiller",                         # midsize
    8:  "PUE_WUE_Chiller",                            # small
    9:  "PUE_WUE_AIRChiller",                         # small
    10: "PUE_WUE_DX",
    11: "PUE_WUE_AE_Immersion_Chiller",               # AE + DLC (浸没)
    12: "PUE_WUE_Immersion_Chiller_Watereconomier",   # WE + DLC (浸没)
}

# --- 基础参数块，对应各函数里真实的 w[i] 顺序 ---

_ORDER_AE_BASE = [                                    # PUE_WUE_AE_Chiller / _Colo / Immersion 系列共用的基础顺序 (29)
    "UPS_e", "PD_lr", "L_percentage", "delta_T_air", "Fan_Pressure_CRAC",
    "Fan_e_CRAC", "Pump_Pressure_HD", "Pump_e_HD", "AT_CT", "Chiller_load",
    "delta_T_water", "Pump_Pressure_CW", "Pump_e_CW", "delta_T_CT",
    "Pump_Pressure_CT", "Pump_e_CT", "Windage_p", "CC", "Fan_Pressure_CT",
    "Fan_e_CT", "SHR", "LGRatio", "T_up", "T_lw", "dp_up", "dp_lw",
    "RH_up", "RH_lw", "pcop",
]

_ORDER_WE = [                                         # PUE_WUE_Chiller_Watereconomier / WE_Chiller_Colo (33)
    "UPS_e", "PD_lr", "L_percentage", "SHR", "delta_T_air", "Fan_Pressure_CRAC",
    "Fan_e_CRAC", "Pump_Pressure_HD", "Pump_e_HD", "HTE", "delta_T_water",
    "AT_CT", "AT_HE", "Pump_Pressure_WE", "Pump_e_WE", "Pump_Pressure_CW",
    "Pump_e_CW", "Chiller_load", "delta_T_CT", "Pump_Pressure_CT", "Pump_e_CT",
    "Windage_p", "CC", "LGRatio", "Fan_Pressure_CT", "Fan_e_CT", "T_up", "T_lw",
    "dp_up", "dp_lw", "RH_up", "RH_lw", "pcop",
]

_ORDER_CHILLER_MID_SMALL = [                          # PUE_WUE_Chiller: Case 5 (midsize) / Case 8 (small) (30)
    "UPS_e", "PD_lr", "L_percentage", "SHR", "delta_T_air",
    "Fan_Pressure_CRAC", "Fan_e_CRAC", "Pump_Pressure_HD", "Pump_e_HD", "HTE",
    "delta_T_water", "Pump_Pressure_CW", "Pump_e_CW", "AT_CT", "Chiller_load",
    "delta_T_CT", "Pump_Pressure_CT", "Pump_e_CT", "Windage_p", "CC",
    "Fan_Pressure_CT", "Fan_e_CT", "LGRatio", "T_up", "T_lw", "dp_up", "dp_lw",
    "RH_up", "RH_lw", "pcop",
]

_ORDER_AE_AIRCHILLER = [                              # PUE_WUE_AE_AIRChiller: Case 6 (21)
    "UPS_e", "PD_lr", "L_percentage", "delta_T_air",
    "Fan_Pressure_CRAC", "Fan_e_CRAC", "SHR", "Pump_Pressure_HD", "Pump_e_HD",
    "HTE", "delta_T_water", "Pump_Pressure_CW", "Pump_e_CW", "pcop",
    "Chiller_load", "T_up", "T_lw", "dp_up", "dp_lw", "RH_up", "RH_lw",
]

_ORDER_AIRCHILLER = [                                 # PUE_WUE_AIRChiller: Case 7 (midsize) / Case 9 (small) (21)
    "UPS_e", "PD_lr", "L_percentage", "SHR", "delta_T_air",
    "Fan_Pressure_CRAC", "Fan_e_CRAC", "Pump_Pressure_HD", "Pump_e_HD", "HTE",
    "delta_T_water", "Pump_Pressure_CW", "Pump_e_CW", "Chiller_load", "pcop",
    "T_up", "T_lw", "dp_up", "dp_lw", "RH_up", "RH_lw",
]

_ORDER_DX = [                                         # PUE_WUE_DX: Case 10 (14)
    "UPS_e", "PD_lr", "L_percentage", "SHR", "delta_T_air",
    "Fan_Pressure_CRAC", "Fan_e_CRAC", "T_up", "T_lw", "dp_up", "dp_lw",
    "RH_up", "RH_lw", "pcop",
]

_IMMERSION_EXTRA = ["rho_CL", "f_CL", "Pump_Pressure_CL", "Pump_e_CL"]   # 浸没冷却液相关 4 个新参数
_WE_EXTRA        = ["HTE", "AT_HE", "Pump_Pressure_WE", "Pump_e_WE"]     # waterside economizer 相关 4 个参数

CASE_PARAM_ORDER = {
    1:  _ORDER_AE_BASE,
    2:  _ORDER_WE,
    3:  _ORDER_AE_BASE,
    4:  _ORDER_WE,
    5:  _ORDER_CHILLER_MID_SMALL,
    6:  _ORDER_AE_AIRCHILLER,
    7:  _ORDER_AIRCHILLER,
    8:  _ORDER_CHILLER_MID_SMALL,
    9:  _ORDER_AIRCHILLER,
    10: _ORDER_DX,
    11: _ORDER_AE_BASE + _IMMERSION_EXTRA,                  # 29 + 4 = 33
    12: _ORDER_AE_BASE + _WE_EXTRA + _IMMERSION_EXTRA,      # 29 + 4 + 4 = 37
}


In [ ]:
def load_ranges(xlsx_path):
    """从新版 Table_B1_对照表_补充DLC.xlsx 的 '代码可用(转换后)' sheet 读取每个 case 每个参数的 (low, high)。
    Case 1-10 是原来的风冷/水冷/DX 系统；Case 11 = Case 1-DLC(AE+浸没)，Case 12 = Case 2-DLC(WE+浸没)。
    返回: ranges[case_id][code_var] = (low, high)"""
    wb = load_workbook(xlsx_path, data_only=True)
    ws = wb[SHEET_NAME]

    ranges = {c: {} for c in range(1, N_CASES + 1)}
    row = FIRST_DATA_ROW
    while True:
        code_var = ws.cell(row=row, column=CODE_VAR_COL).value
        if code_var is None:
            break
        for case_id in range(1, N_CASES + 1):
            col = CASE_COL_START + (case_id - 1)
            val = ws.cell(row=row, column=col).value
            if val is None or str(val).strip().lower() == "n/a":
                continue
            low_str, high_str = str(val).split("–")
            ranges[case_id][code_var] = (float(low_str), float(high_str))
        row += 1
    return ranges


def lhs_sample_case(case_id, ranges, n_samples, seed):
    order = CASE_PARAM_ORDER[case_id]
    case_ranges = ranges[case_id]

    missing = [p for p in order if p not in case_ranges]
    if missing:
        raise ValueError(f"Case {case_id} 缺少参数区间: {missing}，"
                          f"请检查 Excel 表里对应格子是否被误标为 n/a。")

    l_bounds = [case_ranges[p][0] for p in order]
    u_bounds = [case_ranges[p][1] for p in order]

    sampler = qmc.LatinHypercube(d=len(order), seed=seed)
    unit_sample = sampler.random(n=n_samples)          # [0,1) 区间, shape (n_samples, d)
    scaled = qmc.scale(unit_sample, l_bounds, u_bounds)  # 缩放到各参数真实区间(均匀分布)

    return pd.DataFrame(scaled, columns=order)


def main(table="Table_B1_对照表_补充DLC.xlsx",
         n_samples=50,
         seed=42,
         out_dir="lhs_samples"):
    """
    生成每个 Case (1-12) 的 Latin Hypercube Sampling 参数
    """
    os.makedirs(out_dir, exist_ok=True)
    ranges = load_ranges(table)
    for case_id in range(1, N_CASES + 1):
        # 每个 Case 使用不同 seed，避免抽样完全同步
        df = lhs_sample_case(
            case_id=case_id,
            ranges=ranges,
            n_samples=n_samples,
            seed=seed + case_id
        )
        out_path = os.path.join(out_dir, f"case_{case_id}.csv")

        df.to_csv(out_path, index=False)
        print(
            f"Case {case_id:2d} ({CASE_FUNC_NAME[case_id]:<42}) "
            f"-> {n_samples:3d} samples × {df.shape[1]:2d} params "
            f"-> {out_path}"
        )
    print("\n完成！")



main(
    table=str(DATA_ROOT / '4_ThermoDynamicsModel/Table_B1_Final.xlsx'),
    n_samples=50,
    seed=42,
    out_dir=str(DATA_ROOT / '4_ThermoDynamicsModel/HyperCubicSampling')
)
